In [ ]:
pip install biopython ete3 toytree

In [ ]:
from Bio import Phylo
import re

infile = "/content/drive/MyDrive/X"
fixedfile = "/content/drive/MyDrive/X.tree"

with open(infile, "r", encoding="utf-8") as f:
    text = f.read()

# zamena samo vo takson iminja od tip ACCESSION_YYYY-MM-DD -> ACCESSION_YYYY_MM_DD
text = re.sub(r'([A-Za-z0-9]+_\d{4})-(\d{2})-(\d{2})', r'\1_\2_\3', text)

with open(fixedfile, "w", encoding="utf-8") as f:
    f.write(text)

tree = Phylo.read(fixedfile, "nexus")
Phylo.draw_ascii(tree)
print(f"Total tips: {len(tree.get_terminals())}")
print(f"Root comment: {tree.root.comment}")

                                    ____________ EU918764_2007_08_15
                                   |
                                ___|         ___ FJ610151_2008_07_01
                               |   |        |
                               |   |        |    _ LC718901_2021_07_01
                               |   |________|  ,|
                               |            | ,||_ PV012280_2024_01_15
                               |            | ||
                               |            |_||__ LC888264_2022_07_01
                               |              |
                               |              |___ LC888269_2023_07_01
                               |
                               |            ____ FJ006723_2006_12_16
                               |           |
                               |          ,|___ JN565303_2001_07_01
                               |          ||
                               |         ,||____ JQ004093_2010_07_01
                   

In [ ]:
import re

root = tree.root
comment = root.comment or ""

print("Root comment:")
print(comment)

height_match = re.search(r'height=([0-9eE+\-.]+)', comment)
hpd_match = re.search(r'height_95%_HPD=\{([0-9eE+\-.]+),([0-9eE+\-.]+)\}', comment)

if height_match:
    tmrca = float(height_match.group(1))
    print(f"TMRCA: {tmrca}")
else:
    print("TMRCA not found in root comment.")

if hpd_match:
    hpd_low = float(hpd_match.group(1))
    hpd_high = float(hpd_match.group(2))
    print(f"95% HPD: {hpd_low} – {hpd_high}")
else:
    print("95% HPD not found in root comment.")

Root comment:
[&height=315.50960539961534,height_95%_HPD={227.76042241004708,415.11692015196763},height_median=309.6709992853266,height_range={182.64560904226,627.4489652801781},length=0.0,posterior=1.0]
TMRCA: 315.50960539961534
95% HPD: 227.76042241004708 – 415.11692015196763


In [ ]:
import pandas as pd

log_path = "/content/drive/MyDrive/X.log"
log = pd.read_csv(log_path, sep="\t", comment="#")

print(log.columns.tolist())

['Sample', 'posterior', 'likelihood', 'prior', 'treeLikelihood', 'Tree.height', 'Tree.treeLength', 'clockRate', 'kappa', 'proportionInvariant', 'gammaShape', 'CoalescentExponential', 'ePopSize', 'growthRate']


In [ ]:
import pandas as pd

log_path = "/content/drive/MyDrive/X.log"
log = pd.read_csv(log_path, sep="\t", comment="#")

clock = log["clockRate"].dropna()

print(clock.describe())
print(f"Mean: {clock.mean():.6f}")
print(f"95% HPD-like interval (quantiles): {clock.quantile(0.025):.6f} – {clock.quantile(0.975):.6f}")

count    100001.000000
mean          0.000059
std           0.003173
min           0.000018
25%           0.000036
50%           0.000041
75%           0.000046
max           1.000000
Name: clockRate, dtype: float64
Mean: 0.000059
95% HPD-like interval (quantiles): 0.000029 – 0.000073


Следново: ова се quantiles, не exact BEAST HPD interval

In [ ]:
import pandas as pd

log_path = "/content/drive/MyDrive/X.log"
log = pd.read_csv(log_path, sep="\t", comment="#")

burnin = int(len(log) * 0.1)   # 10% burn-in
log_post = log.iloc[burnin:]

clock = log_post["clockRate"].dropna()

print(f"Rows total: {len(log)}")
print(f"Rows after burn-in: {len(log_post)}")
print(clock.describe())
print(f"Mean: {clock.mean():.6f}")
print(f"2.5% quantile: {clock.quantile(0.025):.6f}")
print(f"97.5% quantile: {clock.quantile(0.975):.6f}")

Rows total: 100001
Rows after burn-in: 90001
count    90001.000000
mean         0.000040
std          0.000006
min          0.000018
25%          0.000036
50%          0.000040
75%          0.000044
max          0.000068
Name: clockRate, dtype: float64
Mean: 0.000040
2.5% quantile: 0.000029
97.5% quantile: 0.000053


In [ ]:
!pip install toytree toyplot matplotlib

In [ ]:
import toytree
import toyplot
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

tree_path = "/content/drive/MyDrive/X.tree"
tre = toytree.tree(tree_path)

tips = tre.get_tip_labels()

years = []
for t in tips:
    parts = t.split("_")
    years.append(parts[1] if len(parts) >= 2 else "unknown")

# unique years
unique_years = sorted(set(years))

# zemame boi od matplotlib colormap
cmap = plt.get_cmap("Set2", len(unique_years))
year_to_color = {
    year: mcolors.to_hex(cmap(i))
    for i, year in enumerate(unique_years)
}

colors = [year_to_color[y] for y in years]

canvas, axes, mark = tre.draw(
    width=900,
    height=8000,
    tip_labels=True,
    tip_labels_colors=colors
)

<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="900.0px" height="8000.0px" viewBox="0 0 900.0 8000.0" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="ta3c61a90b08e49379a038f545dc72fb2"> EU918764_2007_08_15 FJ610151_2008_07_01 LC718901_2021_07_01 PV012280_2024_01_15 LC888264_2022_07_01 LC888269_2023_07_01 FJ006723_2006_12_16 JN565303_2001_07_01 JQ004093_2010_07_01 KY549225_2009_07_01 JN565302_2001_07_01 JQ004092_2010_07_01 KY549276_2010_07_01 KY549221_2010_07_01 PV051125_2020_07_01 OP728355_2018_07_01 OP728365_2018_07_01 OP728371_2018_07_01 OP728404_2018_07_01 OP728457_2018_07_01 OP728364_2018_07_01 OP728380_2018_07_01 JQ067943_2003_07_01 KY549166_2008_07_01 KY549248_2011_07_01 KY549200_2008_07_01 KY549211_2009_07_01 KY549164_2008_07_01 KY549160_2008_07_01 KY549167_2008_07_01 KY549288_2009_07_01 LC786757_2021_07_01 PX461755_2022_07_01 PX461759_2022_07_01 KU684316_2009_07_01 OP728354_2018_07_01 OP751554_2016_07_01 OP751676_2016_07_01 OP751616_2016_07_01 OP751624_2016_07_01 OP728421_2018_07_01 OP751586_2016_07_01 OP751620_2016_07_01 OP751651_2016_07_01 OP751601_2016_07_01 OP751678_2016_07_01 OP751582_2016_07_01 OP728384_2018_07_01 OP751612_2016_07_01 OP751613_2016_07_01 OP751653_2016_07_01 KX947276_2014_07_01 KX947278_2014_07_01 KX947282_2014_07_01 KX947281_2014_07_01 KX947277_2014_07_01 PV051146_2024_07_01 KX947279_2014_07_01 KX947280_2014_07_01 KF880690_2011_02_21 KY549207_2009_07_01 PV051165_2021_07_01 KY549191_2008_07_01 KY549321_2009_07_01 PX461753_2022_07_01 PX461757_2022_07_01 PX461758_2022_07_01 PX461760_2022_07_01 KY549184_2008_07_01 KY549277_2008_07_01 KY549161_2008_07_01 KY549254_2009_07_01 KY549188_2008_07_01 KY549258_2010_07_01 KY549245_2009_07_01 PV051138_2024_07_01 KY549295_2010_07_01 KY549313_2009_07_01 KY549193_2008_07_01 KY549214_2009_07_01 KY549230_2010_07_01 KY549198_2008_07_01 OP751562_2016_07_01 OP751559_2016_07_01 OP751623_2016_07_01 OP728350_2018_07_01 OP728362_2018_07_01 OP728361_2018_07_01 OP728377_2018_07_01 OP728415_2018_07_01 OP728458_2018_07_01 OP751563_2016_07_01 OP751575_2016_07_01 OP751576_2016_07_01 OP751684_2016_07_01 OP751652_2016_07_01 OP751654_2016_07_01 OP751669_2016_07_01 OP751607_2016_07_01 OP728389_2018_07_01 OP751647_2016_07_01 OP728396_2018_07_01 OP751552_2016_07_01 OP751636_2016_07_01 OP751609_2016_07_01 OP728387_2018_07_01 OP728412_2018_07_01 OP751658_2016_07_01 OP728382_2018_07_01 FJ610147_2008_07_01 KY549171_2008_07_01 KY549186_2008_07_01 KY549196_2008_07_01 PV051151_2021_07_01 KY549252_2010_07_01 KY549307_2011_07_01 KY549199_2008_07_01 KY549270_2010_07_01 KY549311_2010_07_01 PV051139_2024_07_01 KY549299_2009_07_01 KY549162_2008_07_01 KY549264_2008_07_01 LC888263_2022_07_01 KY549185_2008_07_01 KY549187_2008_07_01 KY549216_2009_07_01 KY549194_2008_07_01 KY549219_2010_07_01 KY549203_2009_07_01 KY549283_2009_07_01 PX461752_2022_07_01 FJ610148_2008_07_01 KY549206_2009_07_01 KY549273_2011_07_01 OP751561_2016_07_01 OP751583_2016_07_01 OP751592_2016_07_01 OP751679_2016_07_01 KY549223_2010_07_01 KY549309_2009_07_01 OP751617_2016_07_01 KU684317_2009_07_01 PV051132_2024_07_01 OP751591_2016_07_01 OP751667_2016_07_01 KY883659_2016_07_01 OP751604_2016_07_01 PV051143_2024_07_01 PV051121_2020_07_01 PV051135_2023_07_01 PV051164_2021_07_01 OP728353_2018_07_01 OP728460_2018_07_01 OP751666_2016_07_01 OP751618_2016_07_01 OP751650_2016_07_01 OP728375_2018_07_01 OP751578_2016_07_01 OP751671_2016_07_01 OP751603_2016_07_01 OP728363_2018_07_01 OP751588_2016_07_01 OP751661_2016_07_01 OP728381_2018_07_01 OP728409_2018_07_01 OP728440_2018_07_01 OP751614_2016_07_01 OP751619_2016_07_01 OP751682_2016_07_01 OP728376_2018_07_01 OP728378_2018_07_01 OP728403_20

In [ ]:
print("Year to color mapping:")
for year, color in year_to_color.items():
    print(year, "->", color)

Year to color mapping:
2001 -> #66c2a5
2003 -> #66c2a5
2004 -> #66c2a5
2006 -> #fc8d62
2007 -> #fc8d62
2008 -> #8da0cb
2009 -> #8da0cb
2010 -> #8da0cb
2011 -> #e78ac3
2012 -> #e78ac3
2013 -> #a6d854
2014 -> #a6d854
2015 -> #a6d854
2016 -> #ffd92f
2017 -> #ffd92f
2018 -> #e5c494
2020 -> #e5c494
2021 -> #e5c494
2022 -> #b3b3b3
2023 -> #b3b3b3
2024 -> #b3b3b3
